# 🏛️ LakehouseRAG

**다중 포맷 지능형 검색 시스템 | BM25 + Fuzzy + Semantic 3중 하이브리드**

---
| 기능 | 내용 |
|------|------|
| 📥 데이터 적재 | TXT/PDF/DOCX/XLSX/CSV/JSON/이미지 + MD5 해시 증분 적재 |
| 🔍 검색 엔진 | BM25 + Fuzzy + Semantic 3중 하이브리드 |
| 🧠 임베딩 | paraphrase-multilingual-MiniLM-L12-v2 (다국어) |
| 💬 LLM | Gemini 2.5 Flash + 슬라이딩 윈도우 대화 메모리 |
| 📊 UI | Streamlit 채팅 + 파일 탐색기 + 현황 대시보드 |

## ① 패키지 설치

In [ ]:
!pip install -q \
    langchain langchain-community langchain-google-genai langchain-huggingface \
    faiss-cpu sentence-transformers \
    rank_bm25 thefuzz python-Levenshtein \
    pymupdf pdfplumber python-docx openpyxl \
    easyocr pillow \
    pandas pyarrow \
    streamlit pyngrok \
    google-generativeai

print('✅ 패키지 설치 완료')

In [ ]:
!pip install -q langchain-classic

## ② Google Drive 마운트 & API 키 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re

API_KEY_PATH  = '/content/drive/MyDrive/DATA_LAKEHOUSE/api_key.txt'
DATA_DIR      = '/content/drive/MyDrive/DATA_LAKEHOUSE/DATA'
LAKEHOUSE_DB  = '/content/drive/MyDrive/DATA_LAKEHOUSE/structured_lakehouse.parquet'
FAISS_DIR     = '/content/faiss_index_db'
HASH_LOG      = '/content/drive/MyDrive/DATA_LAKEHOUSE/file_hash_log.json'

# API 키 파싱
with open(API_KEY_PATH, 'r') as f:
    for line in f:
        m = re.match(r'GOOGLE_API_KEY\s*=\s*(.+)', line.strip())
        if m:
            os.environ['GOOGLE_API_KEY'] = m.group(1).strip()
            break

assert os.environ.get('GOOGLE_API_KEY'), '❌ GOOGLE_API_KEY를 찾을 수 없습니다.'
print(f'✅ API 키 로드 완료 | 데이터 디렉토리: {DATA_DIR}')

## ③ 데이터 수집 모듈 (ingestion)

In [ ]:
import hashlib, json, datetime, warnings
import pandas as pd
import fitz                       # PyMuPDF
import pdfplumber
from docx import Document as DocxDocument
from PIL import Image
import easyocr
warnings.filterwarnings('ignore')

_ocr_reader = None
def get_ocr_reader():
    global _ocr_reader
    if _ocr_reader is None:
        print('🔧 OCR 엔진 초기화 중...')
        _ocr_reader = easyocr.Reader(['ko', 'en'], gpu=False)
    return _ocr_reader


def md5_of_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()


def extract_pdf_tables(path: str) -> str:
    """pdfplumber로 표 추출 → 마크다운 형식 반환"""
    table_texts = []
    try:
        with pdfplumber.open(path) as pdf:
            for i, page in enumerate(pdf.pages):
                tables = page.extract_tables()
                for tbl in tables:
                    if not tbl:
                        continue
                    rows = []
                    for j, row in enumerate(tbl):
                        cells = [str(c or '').replace('\n', ' ') for c in row]
                        rows.append('| ' + ' | '.join(cells) + ' |')
                        if j == 0:
                            rows.append('|' + '---|' * len(cells))
                    table_texts.append(f'[표 - 페이지 {i+1}]\n' + '\n'.join(rows))
    except Exception as e:
        table_texts.append(f'[표 추출 오류: {e}]')
    return '\n\n'.join(table_texts)


def extract_text_from_file(file_path: str) -> str:
    ext = os.path.splitext(file_path)[-1].lower()
    text = ''
    try:
        if ext == '.txt':
            with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
                text = f.read()

        elif ext == '.pdf':
            # 본문 텍스트 (PyMuPDF)
            doc = fitz.open(file_path)
            pages_text = [page.get_text() for page in doc]
            body = '\n'.join(pages_text)
            # 표 텍스트 (pdfplumber)
            tables = extract_pdf_tables(file_path)
            text = body
            if tables.strip():
                text += '\n\n[PDF 표 데이터]\n' + tables

        elif ext == '.docx':
            doc = DocxDocument(file_path)
            paragraphs = [p.text for p in doc.paragraphs]
            table_rows = []
            for table in doc.tables:
                for row in table.rows:
                    table_rows.append(' | '.join(cell.text.strip() for cell in row.cells))
            text = '\n'.join(paragraphs)
            if table_rows:
                text += '\n\n[DOCX 표]\n' + '\n'.join(table_rows)

        elif ext in ['.xlsx', '.xls']:
            xl = pd.read_excel(file_path, sheet_name=None)
            parts = []
            for sheet_name, df in xl.items():
                parts.append(f'[시트: {sheet_name}]\n{df.to_string(index=False)}')
            text = '\n\n'.join(parts)

        elif ext == '.csv':
            df = pd.read_csv(file_path, encoding='utf-8', errors='replace')
            text = df.to_string(index=False)

        elif ext == '.json':
            with open(file_path, 'r', encoding='utf-8') as f:
                text = json.dumps(json.load(f), ensure_ascii=False, indent=2)

        elif ext in ['.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp']:
            reader = get_ocr_reader()
            result = reader.readtext(file_path, detail=0, paragraph=True)
            text = ' '.join(result)

        else:
            text = f'[지원하지 않는 형식: {ext}]'

    except Exception as e:
        text = f'[추출 오류: {e}]'

    return text.strip()


def sync_lakehouse(data_dir: str, db_path: str, hash_log_path: str) -> pd.DataFrame:
    """MD5 해시 기반 증분 적재. 변경된 파일만 재처리."""
    # 기존 DB 로드
    if os.path.exists(db_path):
        print('📦 기존 레이크하우스 로드 중...')
        lakehouse_df = pd.read_parquet(db_path)
    else:
        print('🚀 레이크하우스 신규 생성...')
        lakehouse_df = pd.DataFrame(columns=['file_name','extension','content','path','last_modified','md5'])

    # 해시 로그 로드
    old_hashes = {}
    if os.path.exists(hash_log_path):
        with open(hash_log_path, 'r') as f:
            old_hashes = json.load(f)

    new_hashes = dict(old_hashes)
    new_count = updated_count = skipped_count = 0
    new_rows = []

    supported_ext = {'.txt','.pdf','.docx','.xlsx','.xls','.csv','.json',
                     '.png','.jpg','.jpeg','.bmp','.tiff','.webp'}

    for filename in sorted(os.listdir(data_dir)):
        full_path = os.path.join(data_dir, filename)
        if not os.path.isfile(full_path):
            continue
        ext = os.path.splitext(filename)[-1].lower()
        if ext not in supported_ext:
            continue

        current_hash = md5_of_file(full_path)
        current_mtime = os.path.getmtime(full_path)

        if old_hashes.get(filename) == current_hash:
            skipped_count += 1
            continue

        if filename in old_hashes:
            print(f'  🔄 변경 감지: {filename}')
            lakehouse_df = lakehouse_df[lakehouse_df['file_name'] != filename]
            updated_count += 1
        else:
            print(f'  ✨ 신규 파일: {filename}')
            new_count += 1

        content = extract_text_from_file(full_path)
        new_rows.append({
            'file_name': filename, 'extension': ext, 'content': content,
            'path': full_path, 'last_modified': current_mtime, 'md5': current_hash
        })
        new_hashes[filename] = current_hash

    if new_rows:
        lakehouse_df = pd.concat([lakehouse_df, pd.DataFrame(new_rows)], ignore_index=True)
        lakehouse_df.to_parquet(db_path, index=False)
        with open(hash_log_path, 'w') as f:
            json.dump(new_hashes, f, ensure_ascii=False, indent=2)
        print(f'\n✅ 동기화 완료 — 신규: {new_count} | 업데이트: {updated_count} | 스킵: {skipped_count}')
    else:
        print('✅ 모든 파일이 최신 상태입니다.')

    return lakehouse_df


print('✅ ingestion 모듈 정의 완료')

## ④ 검색 엔진 모듈 (engine) — BM25 + Fuzzy + Semantic

In [ ]:
import re, unicodedata
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from collections import deque

import numpy as np
import pandas as pd
from thefuzz import fuzz
from rank_bm25 import BM25Okapi

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# ── 유틸 ──────────────────────────────────────────────────────────────────────
def is_hangul(text: str) -> bool:
    return bool(re.search('[가-힣]', text))

def decompose_text(text: str) -> str:
    return ''.join(unicodedata.normalize('NFD', text))

def classify_query(query: str, llm=None) -> str:
    """AI 기반 쿼리 유형 분류: 'regex' | 'word' | 'sentence'"""
    classify_prompt = (
        f'다음 검색 입력의 유형을 판단하세요.\n\n'
        f'입력: "{query}"\n\n'
        '유형 정의:\n'
        '- regex: 정규식 패턴 (특수문자 포함, 패턴 검색 의도)\n'
        '- word: 단어/키워드 검색 (짧은 명사, 고유명사, 용어 등. 공백이 있어도 키워드 나열이면 word)\n'
        '- sentence: 자연어 질문 또는 문장 (의문문, 서술문, 설명/분석 요청 등)\n\n'
        "반드시 'regex', 'word', 'sentence' 중 하나만 답하세요. 다른 말은 절대 하지 마세요."
    )
    try:
        if llm is not None:
            response = llm.invoke(classify_prompt)
        else:
            _llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0,
                                           google_api_key=os.environ['GOOGLE_API_KEY'])
            response = _llm.invoke(classify_prompt)
        result = response.content.strip().lower()
        if result in ('regex', 'word', 'sentence'):
            return result
    except Exception:
        pass
    # fallback: LLM 실패 시 규칙 기반
    if any(c in query for c in r'.^$*+?{}[]\\|()'):
        return 'regex'
    if len(query.strip().split()) == 1:
        return 'word'
    return 'sentence'

def get_context_snippet(content: str, target: str, window: int = 80) -> str:
    content_clean = content.replace('\n', ' ')
    idx = content_clean.lower().find(target.lower())
    if idx == -1:
        return content_clean[:window * 2] + '...'
    matched = content_clean[idx: idx + len(target)]
    start = max(0, idx - window)
    end = min(len(content_clean), idx + len(matched) + window)
    snippet = content_clean[start:end]
    highlighted = snippet.replace(matched, f"**{matched}**")
    prefix = '...' if start > 0 else ''
    suffix = '...' if end < len(content_clean) else ''
    return f'{prefix}{highlighted}{suffix}'


# ── BM25 검색 ────────────────────────────────────────────────────────────────
class BM25Retriever:
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        self.corpus = [re.sub(r'\s+', ' ', row['content']).split()
                       for _, row in self.df.iterrows()]
        self.bm25 = BM25Okapi(self.corpus)

    def search(self, query: str, top_k: int = 10) -> pd.DataFrame:
        tokens = query.split()
        scores = self.bm25.get_scores(tokens)
        top_idx = np.argsort(scores)[::-1][:top_k]
        results = []
        for i in top_idx:
            if scores[i] > 0:
                results.append({
                    '파일명': self.df.iloc[i]['file_name'],
                    'BM25점수': round(float(scores[i]), 3),
                    '방식': 'BM25',
                    '미리보기': get_context_snippet(self.df.iloc[i]['content'], tokens[0] if tokens else query)
                })
        return pd.DataFrame(results)


# ── Fuzzy 검색 ───────────────────────────────────────────────────────────────
def fuzzy_search(df: pd.DataFrame, query: str, threshold: int = 45) -> pd.DataFrame:
    results = []
    query_clean = query.lower().strip()
    has_hangul = is_hangul(query_clean)
    query_proc = decompose_text(query_clean)

    # Regex 모드
    is_regex = any(c in query for c in r'.^$*+?{}[]\|()')
    if is_regex:
        try:
            pattern = re.compile(query, re.IGNORECASE)
            for _, row in df.iterrows():
                m = pattern.search(row['content'])
                if m:
                    results.append({'파일명': row['file_name'], '유사도': 100.0,
                                    '매칭단어': m.group(), '방식': 'Regex',
                                    '미리보기': get_context_snippet(row['content'], m.group())})
            return pd.DataFrame(results)
        except Exception:
            pass

    for _, row in df.iterrows():
        words = re.sub(r'[^a-zA-Z0-9가-힣\s]', ' ', row['content'].lower()).split()
        best_score, best_word = 0, 'N/A'
        for word in set(words):
            wp = decompose_text(word) if has_hangul else word
            if has_hangul and len(query_clean) > 1 and len(word) == 1:
                continue
            if not has_hangul and len(query_proc) > 1 and len(wp) == 1:
                continue
            if len(query_proc) >= 3 and (query_proc in wp or wp in query_proc):
                score = 100.0 if query_proc == wp else 90.0
            elif has_hangul:
                score = 100.0 if query_proc == wp else fuzz.ratio(query_proc, wp)
                if query_clean[0] != word[0]: score *= 0.7
            else:
                score = fuzz.ratio(query_proc, wp) * 0.5 + fuzz.token_sort_ratio(query_proc, wp) * 0.5
                if query_clean[0] != word[0]: score *= 0.4
            limit = 55 if not has_hangul else threshold
            if score >= limit and score > best_score:
                best_score, best_word = score, word

        if best_score >= (55 if not has_hangul else threshold):
            results.append({'파일명': row['file_name'], '유사도': round(float(best_score), 1),
                            '매칭단어': best_word, '방식': 'Fuzzy',
                            '미리보기': get_context_snippet(row['content'], best_word)})

    if results:
        return pd.DataFrame(results).sort_values('유사도', ascending=False)
    return pd.DataFrame()


# ── 대화 메모리 ──────────────────────────────────────────────────────────────
@dataclass
class ConversationMemory:
    max_turns: int = 6  # 슬라이딩 윈도우 크기
    history: deque = field(default_factory=deque)

    def add(self, role: str, content: str):
        self.history.append({'role': role, 'content': content})
        while len(self.history) > self.max_turns * 2:
            self.history.popleft()

    def to_langchain_messages(self):
        msgs = []
        for m in self.history:
            if m['role'] == 'user':
                msgs.append(HumanMessage(content=m['content']))
            else:
                msgs.append(AIMessage(content=m['content']))
        return msgs

    def clear(self):
        self.history.clear()

    def to_list(self) -> list:
        return list(self.history)


# ── 벡터 DB ──────────────────────────────────────────────────────────────────
_EMBED_MODEL = None

def get_embeddings():
    global _EMBED_MODEL
    if _EMBED_MODEL is None:
        print('🔧 임베딩 모델 로드 중 (paraphrase-multilingual-MiniLM-L12-v2)...')
        _EMBED_MODEL = HuggingFaceEmbeddings(
            model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
        )
    return _EMBED_MODEL


def build_vector_db(df: pd.DataFrame, faiss_dir: str, hash_log: str) -> FAISS:
    embeddings = get_embeddings()
    current_meta = {r['file_name']: r['md5'] for _, r in df.iterrows()}
    old_meta = {}
    if os.path.exists(hash_log):
        with open(hash_log, 'r') as f:
            old_meta = json.load(f)

    if os.path.exists(faiss_dir) and current_meta == old_meta:
        print('✨ 기존 FAISS 인덱스를 사용합니다.')
        return FAISS.load_local(faiss_dir, embeddings, allow_dangerous_deserialization=True)

    print('🔄 FAISS 인덱스 재구축 중...')
    docs = [
        Document(
            page_content=f"--- FILE: {r['file_name']} ---\n{r['content']}",
            metadata={'file_name': r['file_name'], 'extension': r['extension']}
        )
        for _, r in df.iterrows()
    ]
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = splitter.split_documents(docs)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(faiss_dir)
    print(f'✅ FAISS 인덱스 저장 완료 ({len(chunks)} 청크)')
    return vectorstore


# ── CoT 구조화 프롬프트 ────────────────────────────────────────────────────────
SYSTEM_PROMPT = """당신은 최고 수준의 레이크하우스 데이터 분석 전문가입니다.
아래의 사고 과정(Chain-of-Thought)을 반드시 따르세요:

1. **[문서 파악]** 제공된 문서 목록과 각 파일의 주제를 파악합니다.
2. **[근거 추출]** 질문과 직접 관련된 핵심 내용을 문서에서 정확히 찾아냅니다.
3. **[종합 분석]** 여러 문서의 정보를 교차 검증하고 종합합니다.
4. **[답변 구성]** 명확하고 구조적인 답변을 작성합니다.
5. **[출처 명시]** 답변에 사용된 파일명을 반드시 `📄 출처: [파일명]` 형식으로 표기합니다.

규칙:
- 문서에 없는 내용은 추측하지 마세요. 근거가 없으면 "해당 정보를 문서에서 찾을 수 없습니다"라고 답하세요.
- 특수 패턴(정규식 등)은 키워드 검색 결과를 우선 참조하세요.
- 표나 수치가 있으면 표 형식으로 정리해서 보여주세요.
- 항상 한국어로 답변하세요.

검색 힌트: {routing_hint}

관련 문서:
{context}"""


# ── 메인 엔진 클래스 ──────────────────────────────────────────────────────────
class LakehouseEngine:
    def __init__(self, df: pd.DataFrame, faiss_dir: str, hash_log: str):
        self.df = df
        self.memory = ConversationMemory(max_turns=6)
        self.bm25 = BM25Retriever(df)
        self.vectorstore = build_vector_db(df, faiss_dir, hash_log)
        self.llm = ChatGoogleGenerativeAI(
            model='gemini-2.5-flash',
            temperature=0.1,
            google_api_key=os.environ['GOOGLE_API_KEY']
        )
        self.retriever = self.vectorstore.as_retriever(
            search_type='mmr', search_kwargs={'k': 20, 'fetch_k': 40}
        )
        print('✅ LakehouseEngine 초기화 완료')

    def hybrid_search(self, query: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """BM25 + Fuzzy 결과 반환"""
        bm25_df = self.bm25.search(query, top_k=10)
        fuzzy_df = fuzzy_search(self.df, query)
        return bm25_df, fuzzy_df

    def ask(self, query: str) -> Dict:
        qtype = classify_query(query, self.llm)
        bm25_df = pd.DataFrame()
        fuzzy_df = pd.DataFrame()

        if qtype == 'regex':
            fuzzy_df = fuzzy_search(self.df, query)   # 내부적으로 regex 분기 실행
            hint_files = set(fuzzy_df['파일명'].unique()) if not fuzzy_df.empty else set()
            routing_hint = f"정규식 검색 관련 파일: {', '.join(hint_files)}" if hint_files else '정규식 매칭 없음.'
        elif qtype == 'word':
            fuzzy_df = fuzzy_search(self.df, query)   # fuzzy만
            hint_files = set(fuzzy_df['파일명'].unique()) if not fuzzy_df.empty else set()
            routing_hint = f"단어 Fuzzy 검색 관련 파일: {', '.join(hint_files)}" if hint_files else '단어 매칭 없음.'
        else:  # sentence
            routing_hint = '문장 질문. 의미 기반(Semantic) 검색만 활용.'

        # keyword 결과 → Document 변환
        extra_docs = []
        for df_, col in [(bm25_df, '파일명'), (fuzzy_df, '파일명')]:
            if df_.empty or col not in df_.columns:
                continue
            for _, row in df_.iterrows():
                matched_row = self.df[self.df['file_name'] == row[col]]
                if matched_row.empty:
                    continue
                extra_docs.append(Document(
                    page_content=f"--- FILE: {row[col]} ---\n{row.get('미리보기', '')}",
                    metadata={'file_name': row[col], 'source': row.get('방식', 'keyword')}
                ))

        semantic_docs = self.retriever.invoke(query)

        seen = set()
        all_docs = []
        for doc in extra_docs + semantic_docs:
            key = (doc.metadata.get('file_name', ''), doc.page_content[:100])
            if key not in seen:
                seen.add(key)
                all_docs.append(doc)

        chat_history = self.memory.to_langchain_messages()
        prompt = ChatPromptTemplate.from_messages([
            ('system', SYSTEM_PROMPT),
            MessagesPlaceholder(variable_name='chat_history'),
            ('human', '{input}'),
        ])
        doc_chain = create_stuff_documents_chain(self.llm, prompt)
        result = doc_chain.invoke({
            'input': query,
            'context': all_docs,
            'chat_history': chat_history,
            'routing_hint': routing_hint,
        })

        self.memory.add('user', query)
        self.memory.add('assistant', result)

        return {
            'answer': result,
            'sources': list({doc.metadata.get('file_name', '') for doc in all_docs}),
            'bm25_results': bm25_df,
            'fuzzy_results': fuzzy_df,
            'routing_hint': routing_hint,
            'query_type': qtype,
        }

    def reload(self, new_df: pd.DataFrame, faiss_dir: str, hash_log: str):
        """데이터 리로드 시 엔진 재초기화"""
        self.df = new_df
        self.bm25 = BM25Retriever(new_df)
        self.vectorstore = build_vector_db(new_df, faiss_dir, hash_log)
        self.retriever = self.vectorstore.as_retriever(
            search_type='mmr', search_kwargs={'k': 20, 'fetch_k': 40}
        )
        self.memory.clear()
        print('✅ 엔진 재로드 완료')


print('✅ engine 모듈 정의 완료')

## ⑤ 데이터 적재 실행

In [ ]:
print('=' * 55)
print('  Step 1. 데이터 레이크하우스 동기화')
print('=' * 55)
lakehouse_df = sync_lakehouse(DATA_DIR, LAKEHOUSE_DB, HASH_LOG)

print(f'\n📊 현재 레이크하우스: {len(lakehouse_df)}개 파일')
display(lakehouse_df[['file_name', 'extension', 'last_modified', 'md5']]
        .assign(last_modified=lambda df: pd.to_datetime(df['last_modified'], unit='s')
                                          .dt.strftime('%Y-%m-%d %H:%M:%S'))
        .rename(columns={'file_name':'파일명','extension':'확장자',
                         'last_modified':'최종수정','md5':'MD5'}))

## ⑥ 검색 엔진 초기화

In [ ]:
print('=' * 55)
print('  Step 2. 지능형 검색 엔진 초기화')
print('=' * 55)

engine = LakehouseEngine(lakehouse_df, FAISS_DIR, HASH_LOG)
print('\n🚀 엔진 준비 완료. 검색을 시작할 수 있습니다.')

## ⑦ 대화형 검색 (Colab 인터랙티브)

In [ ]:
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets

# ── UI 컴포넌트 ──
query_input = widgets.Text(
    placeholder='검색어 또는 질문을 입력하세요...',
    layout=widgets.Layout(width='75%')
)
send_btn = widgets.Button(
    description='검색', button_style='primary',
    layout=widgets.Layout(width='10%')
)
clear_btn = widgets.Button(
    description='대화 초기화', button_style='warning',
    layout=widgets.Layout(width='12%')
)
output_area = widgets.Output()

chat_log = []  # {'role': 'user'|'assistant', 'content': str}

def render_chat():
    with output_area:
        clear_output(wait=True)
        for msg in chat_log:
            if msg['role'] == 'user':
                display(HTML(f"""
                <div style='background:#e8f4fd;border-left:4px solid #2196F3;
                            padding:10px 14px;margin:8px 0;border-radius:4px;color:#111'>
                    <b>👤 사용자</b><br>{msg['content']}
                </div>"""))
            else:
                # BM25/Fuzzy 테이블
                qtype = msg.get('query_type', 'sentence')
                if qtype in ('word', 'regex'):
                    label = '🔍 Fuzzy 검색 결과' if qtype == 'word' else '🔎 Regex 검색 결과'
                    if msg.get('fuzzy_df') is not None and not msg['fuzzy_df'].empty:
                        display(HTML(f'<b>{label}:</b>'))
                        display(msg['fuzzy_df'].style.hide(axis='index'))
                # AI 답변
                display(HTML(f"""
                <div style='background:#f0fff4;border-left:4px solid #4CAF50;
                            padding:10px 14px;margin:8px 0;border-radius:4px;color:#111'>
                    <b>🤖 AI 분석 답변</b>
                </div>"""))
                display(Markdown(msg['content']))
                if msg.get('sources'):
                    src_html = ' '.join(f"<span style='background:#e3f2fd;padding:2px 8px;"
                                        f"border-radius:12px;font-size:0.85em'>📄 {s}</span>"
                                        for s in msg['sources'])
                    display(HTML(f'<div style="margin-top:6px; color:#212121;">참조 파일: {src_html}</div>'))

def on_send(b):
    query = query_input.value.strip()
    if not query:
        return
    # 종료 명령 처리
    if query.lower() in ['q', 'exit', 'quit']:
        query_input.value = ''
        with output_area:
            display(HTML("<div style='color:red;font-weight:bold'>🔴 검색을 종료합니다.</div>"))
        send_btn.disabled = True
        query_input.disabled = True
        return
    query_input.value = ''
    chat_log.append({'role': 'user', 'content': query})

    with output_area:
        clear_output(wait=True)
        render_chat()
        display(HTML("<i>⏳ 분석 중...</i>"))

    result = engine.ask(query)
    chat_log.append({
        'role': 'assistant',
        'content': result['answer'],
        'sources': result['sources'],
        'bm25_df': result['bm25_results'],
        'fuzzy_df': result['fuzzy_results'],
        'query_type': result['query_type'],
    })
    render_chat()

def on_clear(b):
    engine.memory.clear()
    chat_log.clear()
    with output_area:
        clear_output()
    display(HTML('<i>✅ 대화 초기화 완료</i>'))

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
query_input.on_submit(on_send)

display(widgets.HBox([query_input, send_btn, clear_btn]))
display(output_area)

## ⑧ Streamlit 앱 작성

In [ ]:
!pip install streamlit

In [ ]:
%%writefile app.py
import os, sys, re, json, hashlib, datetime, unicodedata, warnings
from collections import deque
from dataclasses import dataclass, field
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import streamlit as st
from thefuzz import fuzz
from rank_bm25 import BM25Okapi

warnings.filterwarnings('ignore')

# ── 경로 설정 ─────────────────────────────────────────────────────────────────
API_KEY_PATH = '/content/drive/MyDrive/DATA_LAKEHOUSE/api_key.txt'
DATA_DIR     = '/content/drive/MyDrive/DATA_LAKEHOUSE/DATA'
LAKEHOUSE_DB = '/content/drive/MyDrive/DATA_LAKEHOUSE/structured_lakehouse.parquet'
FAISS_DIR    = '/content/faiss_index_db'
HASH_LOG     = '/content/drive/MyDrive/DATA_LAKEHOUSE/file_hash_log.json'

# ── API 키 로드 ───────────────────────────────────────────────────────────────
def load_api_key():
    with open(API_KEY_PATH, 'r') as f:
        for line in f:
            m = re.match(r'GOOGLE_API_KEY\s*=\s*(.+)', line.strip())
            if m:
                return m.group(1).strip()
    raise ValueError('GOOGLE_API_KEY not found')

os.environ['GOOGLE_API_KEY'] = load_api_key()

# ── 임포트 (무거운 것은 캐시) ─────────────────────────────────────────────────
@st.cache_resource(show_spinner='임베딩 모델 로딩 중...')
def get_embeddings():
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(
        model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    )

@st.cache_resource(show_spinner='OCR 엔진 초기화 중...')
def get_ocr():
    import easyocr
    return easyocr.Reader(['ko', 'en'], gpu=False)

# ── 텍스트 추출 ───────────────────────────────────────────────────────────────
def md5_of_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''): h.update(chunk)
    return h.hexdigest()

def extract_pdf_tables(path):
    import pdfplumber
    rows_md = []
    try:
        with pdfplumber.open(path) as pdf:
            for i, page in enumerate(pdf.pages):
                for tbl in page.extract_tables() or []:
                    rows = []
                    for j, row in enumerate(tbl):
                        cells = [str(c or '').replace('\n', ' ') for c in row]
                        rows.append('| ' + ' | '.join(cells) + ' |')
                        if j == 0: rows.append('|' + '---|' * len(cells))
                    rows_md.append(f'[표 p{i+1}]\n' + '\n'.join(rows))
    except: pass
    return '\n\n'.join(rows_md)

def extract_text(file_path):
    ext = os.path.splitext(file_path)[-1].lower()
    try:
        if ext == '.txt':
            with open(file_path, 'r', encoding='utf-8', errors='replace') as f: return f.read()
        elif ext == '.pdf':
            import fitz
            doc = fitz.open(file_path)
            body = '\n'.join(p.get_text() for p in doc)
            tables = extract_pdf_tables(file_path)
            return body + ('\n\n[표]\n' + tables if tables.strip() else '')
        elif ext == '.docx':
            from docx import Document as D
            doc = D(file_path)
            body = '\n'.join(p.text for p in doc.paragraphs)
            tbls = [' | '.join(c.text for c in row.cells) for t in doc.tables for row in t.rows]
            return body + ('\n\n[표]\n' + '\n'.join(tbls) if tbls else '')
        elif ext in ('.xlsx', '.xls'):
            xl = pd.read_excel(file_path, sheet_name=None)
            return '\n\n'.join(f'[{n}]\n{df.to_string(index=False)}' for n, df in xl.items())
        elif ext == '.csv':
            return pd.read_csv(file_path, encoding='utf-8', errors='replace').to_string(index=False)
        elif ext == '.json':
            with open(file_path, 'r', encoding='utf-8') as f:
                return json.dumps(json.load(f), ensure_ascii=False, indent=2)
        elif ext in ('.png','.jpg','.jpeg','.bmp','.tiff','.webp'):
            return ' '.join(get_ocr().readtext(file_path, detail=0, paragraph=True))
    except Exception as e:
        return f'[오류: {e}]'
    return '[지원하지 않는 형식]'

# ── 레이크하우스 동기화 ───────────────────────────────────────────────────────
@st.cache_data(show_spinner='레이크하우스 동기화 중...')
def load_lakehouse(_trigger: int = 0):
    supported = {'.txt','.pdf','.docx','.xlsx','.xls','.csv','.json',
                 '.png','.jpg','.jpeg','.bmp','.tiff','.webp'}
    df = pd.read_parquet(LAKEHOUSE_DB) if os.path.exists(LAKEHOUSE_DB) else pd.DataFrame(
        columns=['file_name','extension','content','path','last_modified','md5'])
    old_hashes = json.load(open(HASH_LOG)) if os.path.exists(HASH_LOG) else {}
    new_hashes = dict(old_hashes)
    new_rows, log = [], []
    for fn in sorted(os.listdir(DATA_DIR)):
        fp = os.path.join(DATA_DIR, fn)
        if not os.path.isfile(fp): continue
        ext = os.path.splitext(fn)[-1].lower()
        if ext not in supported: continue
        h = md5_of_file(fp)
        if old_hashes.get(fn) == h: continue
        df = df[df['file_name'] != fn]
        new_rows.append({'file_name':fn,'extension':ext,'content':extract_text(fp),
                         'path':fp,'last_modified':os.path.getmtime(fp),'md5':h})
        new_hashes[fn] = h
        log.append(fn)
    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
        df.to_parquet(LAKEHOUSE_DB, index=False)
        json.dump(new_hashes, open(HASH_LOG,'w'), ensure_ascii=False, indent=2)
    return df, log

# ── 벡터 DB ───────────────────────────────────────────────────────────────────
def classify_query(query: str) -> str:
    """AI 기반 쿼리 유형 분류: 'regex' | 'word' | 'sentence'"""
    from langchain_google_genai import ChatGoogleGenerativeAI
    classify_prompt = (
        f'다음 검색 입력의 유형을 판단하세요.\n\n'
        f'입력: "{query}"\n\n'
        '유형 정의:\n'
        '- regex: 정규식 패턴 (특수문자 포함, 패턴 검색 의도)\n'
        '- word: 단어/키워드 검색 (짧은 명사, 고유명사, 용어 등. 공백이 있어도 키워드 나열이면 word)\n'
        '- sentence: 자연어 질문 또는 문장 (의문문, 서술문, 설명/분석 요청 등)\n\n'
        "반드시 'regex', 'word', 'sentence' 중 하나만 답하세요. 다른 말은 절대 하지 마세요."
    )
    try:
        _llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0,
                                       google_api_key=os.environ['GOOGLE_API_KEY'])
        response = _llm.invoke(classify_prompt)
        result = response.content.strip().lower()
        if result in ('regex', 'word', 'sentence'):
            return result
    except Exception:
        pass
    # fallback: LLM 실패 시 규칙 기반
    if any(c in query for c in r'.^$*+?{}[]\\|()'):
        return 'regex'
    if len(query.strip().split()) == 1:
        return 'word'
    return 'sentence'

@st.cache_resource(show_spinner='FAISS 인덱스 빌드 중...')
def get_vectorstore(_trigger: int = 0):
    from langchain_community.vectorstores import FAISS
    from langchain_core.documents import Document
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    df, _ = load_lakehouse(_trigger)
    emb = get_embeddings()
    docs = [Document(page_content=f"--- FILE: {r['file_name']} ---\n{r['content']}",
                     metadata={'file_name':r['file_name'],'extension':r['extension']})
            for _,r in df.iterrows()]
    chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(docs)
    vs = FAISS.from_documents(chunks, emb)
    vs.save_local(FAISS_DIR)
    return vs

# ── 검색 ──────────────────────────────────────────────────────────────────────
def is_hangul(t): return bool(re.search('[가-힣]', t))
def decompose(t): return ''.join(unicodedata.normalize('NFD', t))
def snippet(content, target, w=80):
    cc = content.replace('\n',' ')
    idx = cc.lower().find(target.lower())
    if idx == -1: return cc[:w*2]+'...'
    m = cc[idx:idx+len(target)]
    s,e = max(0,idx-w), min(len(cc),idx+len(m)+w)
    return ('...' if s>0 else '')+cc[s:e].replace(m,f'**{m}**')+('...' if e<len(cc) else '')

def bm25_search(df, query, k=10):
    corpus = [re.sub(r'\s+',' ',r['content']).split() for _,r in df.iterrows()]
    bm = BM25Okapi(corpus)
    scores = bm.get_scores(query.split())
    idx = np.argsort(scores)[::-1][:k]
    return pd.DataFrame([{'파일명':df.iloc[i]['file_name'],'BM25점수':round(float(scores[i]),3),
                           '방식':'BM25','미리보기':snippet(df.iloc[i]['content'],query.split()[0])}
                          for i in idx if scores[i]>0])

def fuzzy_search(df, query, threshold=45):
    results=[]; qc=query.lower().strip(); hg=is_hangul(qc); qp=decompose(qc)
    is_re=any(c in query for c in r'.^$*+?{}[]\|()')
    if is_re:
        try:
            pat=re.compile(query,re.IGNORECASE)
            for _,row in df.iterrows():
                m=pat.search(row['content'])
                if m: results.append({'파일명':row['file_name'],'유사도':100.0,'매칭단어':m.group(),'방식':'Regex','미리보기':snippet(row['content'],m.group())})
            return pd.DataFrame(results)
        except: pass
    for _,row in df.iterrows():
        words=re.sub(r'[^a-zA-Z0-9가-힣\s]',' ',row['content'].lower()).split()
        bs,bw=0,'N/A'
        for w in set(words):
            wp=decompose(w) if hg else w
            if hg and len(qc)>1 and len(w)==1: continue
            if not hg and len(qp)>1 and len(wp)==1: continue
            if len(qp)>=3 and (qp in wp or wp in qp): sc=100.0 if qp==wp else 90.0
            elif hg: sc=(100.0 if qp==wp else fuzz.ratio(qp,wp)); sc*=(0.7 if qc[0]!=w[0] else 1)
            else: sc=fuzz.ratio(qp,wp)*.5+fuzz.token_sort_ratio(qp,wp)*.5; sc*=(0.4 if qc[0]!=w[0] else 1)
            lim=55 if not hg else threshold
            if sc>=lim and sc>bs: bs,bw=sc,w
        if bs>=(55 if not hg else threshold):
            results.append({'파일명':row['file_name'],'유사도':round(float(bs),1),'매칭단어':bw,'방식':'Fuzzy','미리보기':snippet(row['content'],bw)})
    return pd.DataFrame(results).sort_values('유사도',ascending=False) if results else pd.DataFrame()

# ── LLM 답변 ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """당신은 최고 수준의 레이크하우스 데이터 분석 전문가입니다.
Chain-of-Thought 사고 과정:
1. [문서 파악] 제공된 문서 목록과 각 파일의 주제를 파악합니다.
2. [근거 추출] 질문과 관련된 핵심 내용을 정확히 찾아냅니다.
3. [종합 분석] 여러 문서 정보를 교차 검증하고 종합합니다.
4. [답변 구성] 명확하고 구조적인 답변을 작성합니다.
5. [출처 명시] 답변에 사용된 파일명을 `📄 출처: [파일명]` 형식으로 표기합니다.

규칙:
- 문서에 없는 내용은 추측하지 마세요.
- 표/수치는 마크다운 표로 정리하세요.
- 한국어로 답변하세요.

검색 힌트: {routing_hint}

관련 문서:
{context}"""

def get_llm_answer(query, history, vs, df, routing_hint, bm25_df, fuzzy_df):
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_core.messages import HumanMessage, AIMessage
    from langchain_core.documents import Document

    llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.1,
                                  google_api_key=os.environ['GOOGLE_API_KEY'])
    retriever = vs.as_retriever(search_type='mmr', search_kwargs={'k':20,'fetch_k':40})
    chat_history = [HumanMessage(content=m['content']) if m['role']=='user'
                    else AIMessage(content=m['content']) for m in history]

    # BM25/Fuzzy → Document 변환
    extra_docs = []
    for df_, col in [(bm25_df, '파일명'), (fuzzy_df, '파일명')]:
        if df_.empty or col not in df_.columns:
            continue
        for _, row in df_.iterrows():
            matched = df[df['file_name'] == row[col]]
            if matched.empty:
                continue
            extra_docs.append(Document(
                page_content=f"--- FILE: {row[col]} ---\n{row.get('미리보기', '')}",
                metadata={'file_name': row[col], 'source': row.get('방식', 'keyword')}
            ))

    # FAISS 결과
    semantic_docs = retriever.invoke(query)

    # 합치고 중복 제거
    seen = set()
    all_docs = []
    for doc in extra_docs + semantic_docs:
        key = (doc.metadata.get('file_name', ''), doc.page_content[:100])
        if key not in seen:
            seen.add(key)
            all_docs.append(doc)

    prompt = ChatPromptTemplate.from_messages([
        ('system', SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '{input}'),
    ])
    doc_chain = create_stuff_documents_chain(llm, prompt)
    result = doc_chain.invoke({
        'input': query,
        'context': all_docs,
        'chat_history': chat_history,
        'routing_hint': routing_hint,
    })

    sources = list({doc.metadata.get('file_name', '') for doc in all_docs})
    return result, sources
# ── Streamlit UI ──────────────────────────────────────────────────────────────
st.set_page_config(page_title='LakehouseRAG', page_icon='🏛️', layout='wide')

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@400;700&family=Space+Mono:wght@700&display=swap');
html, body, [class*="css"] { font-family: 'Noto Sans KR', sans-serif; }
h1 { font-family: 'Space Mono', monospace; }
.stTabs [data-baseweb="tab"] { font-size: 1rem; font-weight: 700; }
.chat-user { background:#dbeafe; border-left:4px solid #3b82f6;
             padding:12px 16px; border-radius:8px; margin:8px 0; }
.chat-bot  { background:#dcfce7; border-left:4px solid #22c55e;
             padding:12px 16px; border-radius:8px; margin:8px 0; }
.source-badge { background:#e0e7ff; color:#3730a3; padding:2px 10px;
                border-radius:12px; font-size:0.8rem; margin:2px; display:inline-block; }
</style>
""", unsafe_allow_html=True)

st.title('🏛️ LakehouseRAG')
st.caption('BM25 + Fuzzy + Semantic 3중 하이브리드 지능형 검색 시스템')

# 세션 상태 초기화
if 'chat_history' not in st.session_state: st.session_state.chat_history = []
if 'reload_trigger' not in st.session_state: st.session_state.reload_trigger = 0

df, updated_files = load_lakehouse(st.session_state.reload_trigger)
vs = get_vectorstore(st.session_state.reload_trigger)

# ── 탭 ────────────────────────────────────────────────────────────────────────
tab_chat, tab_files, tab_dash = st.tabs(['💬 채팅 검색', '📁 파일 탐색기', '📊 현황 대시보드'])

# ── 탭 1: 채팅 ────────────────────────────────────────────────────────────────
with tab_chat:
    col_chat, col_sidebar = st.columns([3, 1])
    with col_sidebar:
        st.markdown('**⚙️ 설정**')
        if st.button('🔄 데이터 새로고침', use_container_width=True):
            st.session_state.reload_trigger += 1
            st.cache_data.clear(); st.cache_resource.clear()
            st.rerun()
        if st.button('🗑️ 대화 초기화', use_container_width=True):
            st.session_state.chat_history = []
            st.rerun()
        st.divider()
        st.markdown(f'**📂 적재 파일 수:** {len(df)}')
        st.markdown(f'**💾 DB 크기:** {os.path.getsize(LAKEHOUSE_DB)/1024:.1f} KB' if os.path.exists(LAKEHOUSE_DB) else '')

    with col_chat:
        # 채팅 히스토리 렌더링
        for msg in st.session_state.chat_history:
            if msg['role'] == 'user':
                st.markdown(f'<div class="chat-user">👤 <b>사용자</b><br>{msg["content"]}</div>',
                            unsafe_allow_html=True)
            else:
                qtype = msg.get('query_type', 'sentence')
                if qtype in ('word', 'regex'):
                    label = '🔍 Fuzzy 검색 결과' if qtype == 'word' else '🔎 Regex 검색 결과'
                    if msg.get('fuzzy') is not None and not msg['fuzzy'].empty:
                        with st.expander(label, expanded=False):
                            st.dataframe(msg['fuzzy'], use_container_width=True)
                st.markdown(f'<div class="chat-bot">🤖 <b>AI 분석 답변</b></div>',
                            unsafe_allow_html=True)
                st.markdown(msg['content'])
                if msg.get('sources'):
                    badges = ' '.join(f'<span class="source-badge">📄 {s}</span>' for s in msg['sources'])
                    st.markdown(f'참조: {badges}', unsafe_allow_html=True)
                st.divider()

        # 입력
        query = st.chat_input('검색어 또는 질문을 입력하세요...')
        if query:
            st.session_state.chat_history.append({'role':'user','content':query})
            # 슬라이딩 윈도우 (최근 12 메시지)
            window = st.session_state.chat_history[-12:]
            hist_for_llm = [m for m in window if m['role'] in ('user','assistant')]
            # 하이브리드 검색
            qtype = classify_query(query)
            bm25_df = pd.DataFrame()
            fuzzy_df = pd.DataFrame()

            if qtype == 'regex':
                fuzzy_df = fuzzy_search(df, query)
                hint_files = set(fuzzy_df['파일명'].unique()) if not fuzzy_df.empty else set()
                routing_hint = f"정규식 검색 관련 파일: {', '.join(hint_files)}" if hint_files else '정규식 매칭 없음.'
            elif qtype == 'word':
                fuzzy_df = fuzzy_search(df, query)
                hint_files = set(fuzzy_df['파일명'].unique()) if not fuzzy_df.empty else set()
                routing_hint = f"단어 Fuzzy 검색 관련 파일: {', '.join(hint_files)}" if hint_files else '단어 매칭 없음.'
            else:  # sentence
                routing_hint = '문장 질문. 의미 기반(Semantic) 검색만 활용.'

            with st.spinner('🤖 AI가 분석 중입니다...'):
                answer, sources = get_llm_answer(query, hist_for_llm, vs, df, routing_hint, bm25_df, fuzzy_df)
            st.session_state.chat_history.append({
                'role':'assistant','content':answer,'sources':sources,
                'bm25':bm25_df,'fuzzy':fuzzy_df,'query_type':qtype
            })
            st.rerun()

# ── 탭 2: 파일 탐색기 ─────────────────────────────────────────────────────────
with tab_files:
    st.subheader('📁 레이크하우스 파일 탐색기')
    if df.empty:
        st.info('데이터가 없습니다. 데이터 디렉토리를 확인하세요.')
    else:
        ext_filter = st.multiselect('확장자 필터', options=sorted(df['extension'].unique()),
                                     default=sorted(df['extension'].unique()))
        search_filter = st.text_input('파일명 검색')
        filtered = df[df['extension'].isin(ext_filter)]
        if search_filter:
            filtered = filtered[filtered['file_name'].str.contains(search_filter, case=False, na=False)]
        display_df = filtered[['file_name','extension','last_modified','md5']].copy()
        display_df['last_modified'] = pd.to_datetime(display_df['last_modified'], unit='s').dt.strftime('%Y-%m-%d %H:%M')
        display_df.columns = ['파일명','확장자','최종수정','MD5']
        st.dataframe(display_df, use_container_width=True, height=400)

        st.divider()
        selected_file = st.selectbox('파일 내용 미리보기', options=filtered['file_name'].tolist())
        if selected_file:
            content = filtered[filtered['file_name']==selected_file].iloc[0]['content']
            st.text_area('내용 (처음 3000자)', value=content[:3000], height=300)

# ── 탭 3: 대시보드 ────────────────────────────────────────────────────────────
with tab_dash:
    st.subheader('📊 레이크하우스 현황 대시보드')
    if df.empty:
        st.info('데이터가 없습니다.')
    else:
        c1, c2, c3, c4 = st.columns(4)
        c1.metric('📁 총 파일 수', len(df))
        c2.metric('📝 총 문자 수', f"{df['content'].str.len().sum():,}")
        c3.metric('💾 DB 크기', f"{os.path.getsize(LAKEHOUSE_DB)/1024:.1f} KB" if os.path.exists(LAKEHOUSE_DB) else '-')
        c4.metric('🔄 이번 실행 업데이트', len(updated_files))

        st.divider()
        col_a, col_b = st.columns(2)
        with col_a:
            st.markdown('**확장자별 파일 수**')
            ext_cnt = df['extension'].value_counts().reset_index()
            ext_cnt.columns = ['확장자','파일 수']
            st.bar_chart(ext_cnt.set_index('확장자'))
        with col_b:
            st.markdown('**파일별 텍스트 길이 (Top 15)**')
            top15 = df.assign(길이=df['content'].str.len()).nlargest(15,'길이')[['file_name','길이']]
            top15.columns = ['파일명','텍스트 길이']
            st.bar_chart(top15.set_index('파일명'))

        if updated_files:
            st.divider()
            st.markdown('**이번 실행에서 업데이트된 파일:**')
            for f_name in updated_files:
                st.markdown(f'- 🔄 `{f_name}`')


## ⑨ Streamlit 실행

In [ ]:
!pkill -f streamlit
!pkill -f cloudflared

!nohup streamlit run app.py \
  --server.port 8501 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  > streamlit.log 2>&1 &

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

!nohup ./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1 &

In [ ]:
import time
time.sleep(5)

!for i in {1..10}; do \
  url=$(grep -Eo "https://[^ ]+trycloudflare\.com" tunnel.log | head -n 1); \
  if [ -n "$url" ]; then echo "🔗 LAKEHOUSE: $url"; break; fi; \
  sleep 2; \
done

In [ ]:
!ps -ef | grep streamlit | grep -v grep
!ps -ef | grep cloudflared | grep -v grep

In [ ]:
# !pkill -f streamlit
# !pkill -f cloudflared